<a href="https://colab.research.google.com/github/mbenedicto99/Natural-Language-Processing/blob/main/PECE_2026_POS_tagging_e_depend%C3%AAncias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Marcos Lopes (PECE/USP) -- 2026

# Mac-Morpho

Corpus de sentenças cuja fonte são notícias da Folha de S. Paulo no ano de 1994. Etiquetado morfossintaticamente por humanos (padrão ouro).

In [ ]:
# Download do corpus

import nltk
nltk.download('mac_morpho')

In [ ]:
# Uso das sentenças não etiquetadas (como se fosse um corpus de texto "puro")

sents = nltk.corpus.mac_morpho.sents()
sents[:2]

In [ ]:
# Palavras etiquetadas

palavras_etiquetas = nltk.corpus.mac_morpho.tagged_words()
print(palavras_etiquetas[:5])
print(len(palavras_etiquetas))

# Criação de um etiquetador morfossintático (POS-tagger)

Impressione seus amigos com um POS-tagger criado inteiramente por você -- e com três linhas de código!

Antes de começar, vamos aprender a usar o método `get()` dos dicionários do Python. Ele oferece dois recursos importantes:

* Quando se consulta uma chave inexistente, ele não retorna um erro;
* O método permite que seja declarado um valor por default que será automaticamente retornado quando são buscadas chaves inexistentes.

In [ ]:
d = {'abacate': 'n', 'beber': 'v'}

In [ ]:
d['beber']

In [ ]:
# Lembre-se: chamar uma chave inexistente no dicionário dá erro
d['uva']

In [ ]:
# Uma solução para o erro de chave é usar o método dict.get()

d = {'abacate': 'n', 'beber': 'v'}
print(d.get('beber', 'Chave desconhecida.'))
print(d.get('uva', 'Chave desconhecida.'))

In [ ]:
# Criação do dicionário de etiquetas por Compreensão de Dicionário (não de lista)

dic_pos = {palavra.lower(): pos for (palavra, pos) in palavras_etiquetas}
print(len(dic_pos))
dic_pos['média']

In [ ]:
# Criação de uma função de etiquetagem simplificada,
# baseada no dicionário de etiquetas

def tag(sentenca: str) -> list:
    palavras = sentenca.lower().split()
    return [(p, dic_pos.get(p, '<??>')) for p in palavras]

In [ ]:
# Experimente!

tag('Não consigo pensar em uma sentença com uma palavra que não exista no meio daquele um milhão de palavras')

# POS-tagging com spaCy

In [ ]:
# Instalação/atualização do spaCy e dos modelos de língua
!pip install -U spacy

!python -m spacy download pt_core_news_sm  # Notar o "sm": Small Model

In [ ]:
# Importação do módulo e do modelo de língua
import spacy
nlp = spacy.load('pt_core_news_sm')

In [ ]:
# Documento de exemplo
texto = 'Águas passadas não movem moinhos'

In [ ]:
# Outro exemplo bobo, mas que mostra que a POS de uma palavra (ex. "francês")
# é alterada conforme o contexto (substantivo ou adjetivo)

texto = 'O pão francês não foi entregue hoje.'

In [ ]:
doc = nlp(texto)

In [ ]:
for token in doc:
    print(token.text, token.pos_)

# Análise de Dependências

In [ ]:
doc = nlp('João chutou a bola.')
for token in doc:
    print(token.text, token.dep_)

In [ ]:
# Exibição gráfica das dependências
from spacy import displacy

displacy.render(doc, style='dep', jupyter=True)

In [ ]:
# Caso você queira gravar a imagem gerada em arquivo...
from pathlib import Path

svg = displacy.render(doc, style='dep', jupyter=False)
output_path = Path('Minha Sentença.svg')
lixo = output_path.open('w', encoding='utf-8').write(svg)

# Exemplos de aplicação

Esses exemplos vão servir de base para a solução da tarefa de Linguística Forense.

In [ ]:
import pandas as pd

In [ ]:
# Opcional: Barras de progresso (para você não achar que a execução travou...)

from tqdm.notebook import tqdm
tqdm.pandas()

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize


def ler(nome_arquivo) -> str:
    with open(nome_arquivo) as arquivo:
        return arquivo.read()


# Tokenização de palavras
def tokenizar(txt: str) -> list:
    return word_tokenize(txt, language='portuguese')


# Tokenização de sentenças
def tokenizar_sentencas(txt: str) -> list:
    txt = txt.replace('\n', ' ')
    return sent_tokenize(txt, language='portuguese')


# Limpeza simples
def limpar(lista: list) -> list:
    return [i.lower() for i in lista if i.isalpha()]

In [ ]:
texto = ler('Guarani.txt')

In [ ]:
sents = tokenizar_sentencas(texto)

In [ ]:
df_alencar = pd.DataFrame({'Sentencas': sents})
df_alencar.head()

In [ ]:
# O que acontece quando se aplica describe() a strings?

df_alencar.describe()

In [ ]:
# E quanto a contagens de valores?

df_alencar['Sentencas'].value_counts()

In [ ]:
# Um DataFrame tem seus próprios métodos para strings.
# Seu processamento é mais veloz que os equivalentes em Python (compreensão de listas, por ex.)

df_alencar['TamSentencas'] = df_alencar['Sentencas'].str.len()
df_alencar.head()

In [ ]:
# Mais moleza para a sua vida: função que recebe um nome de arquivo
# e retorna um DataFrame de sentenças com seus tamanhos

def textos_df(arq: str) -> pd.DataFrame:
    txt = ler(arq)
    sentencas = tokenizar_sentencas(txt)
    df_obra = pd.DataFrame({'Sentencas': sentencas})
    df_obra['TamSentencas'] = df_obra['Sentencas'].str.len()

    return df_obra

In [ ]:
# Exemplos de aplicação

df_nabuco = textos_df('Abolicionismo.txt')
df_nabuco.head()

In [ ]:
# Uma visualização rápida dos dados numéricos do DataFrame através de um Box Plot

df_alencar.plot.box();

In [ ]:
# Função que recebe uma string e retorna o número de advérbios encontrados nela

def n_advs(sent: str) -> list:
    doc = nlp(sent)
    return len([t.pos_ for t in doc if t.pos_ == 'ADV'])

In [ ]:
n_advs('Teste de não dar nunca nada certo certamente.')

In [ ]:
# Aplicação da função a um df

df_alencar['Advs'] = df_alencar['Sentencas'].progress_apply(n_advs)
df_alencar.sample()  # Para exemplificar

In [ ]:
# Box Plot para os advérbios (semelhante aos de tamanhos de sentenças)

df_alencar['Advs'].plot.box();

In [ ]:
# O módulo matplotlib oferece mais poder e versatilidade com gráficos que o Pandas
# Com ele, fica fácil gerar gráficos diretamente de listas, como faremos a seguir

import matplotlib.pyplot as plt

In [ ]:
df_nabuco.head()

In [ ]:
# Vamos acrescentar uma coluna de advérbios aos demais DFs

df_nabuco['Advs'] = df_nabuco['Sentencas'].progress_apply(n_advs)

In [ ]:
# Agora, será possível compará-los por box plots

adverbios = [i['Advs'].values.tolist() for i in [df_alencar, df_nabuco]]
plt.boxplot(adverbios);

# Tarefa: Linguística Forense

Seu grupo de especialistas em PLN foi chamado para periciar um livro escrito por um autor incógnito. Há três elementos suspeitos de ser o autor da obra. Todos negam veementemente a autoria, mas sabe-se que o autor é um deles. Os suspeitos são:

* Machado de Assis, autor de "Dom Casmurro"
* José de Alencar, autor de "O Guarani"
* Joaquim Nabuco, autor de "O Abolicionismo"

Você tem à disposição esses três livros para comparar com o livro apócrifo.

Em suas análises, use alguns dos recursos comuns da Linguística Forense:

* Análise das divisões de período. Você pode usar somente o tamanho das sentenças, para simplificar, mas uma análise mais completa incluiria todos os delimitadores (vírgula, ponto-e-vírgula, dois pontos, travessão, parênteses...);
* Comparação das colocações adverbiais, em especial os modais (etiqueta de dependência: "advmod") e as conjunções subordinativas ("que" e "se"; etiqueta: "sconj").

A solução do problema deve ser representada num gráfico com duas ou três dimensões, em que cada dimensão expressa uma métrica de comparação, como divisão de período / advérbios modais, por exemplo.

Ao final, responda: quem é o autor do livro apócrifo?